# EvolveGCN-H Training Notebook

**Cell order:**
1. Imports
2. Utility functions
3. GCN norm + WeightEvolver + EvolveGCNHLayer
4. EvolveGCNH model + Splits dataclass
5. run_sequence + evaluate_auroc + evaluate_f1 + train_epoch
6. Config + data loading + model init
7. Training loop + test evaluation

---
## Cell 1 — Imports

In [ ]:
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, roc_auc_score

import sys
print('Python executable:', sys.executable)
print('PyTorch version  :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())

---
## Cell 2 — Utility Functions

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def load_data(data_path: Path):
    return torch.load(data_path, weights_only=False)


def move_snapshots_to_device(snapshots: List, device: torch.device) -> List:
    return [data.to(device) for data in snapshots]


def compute_class_weights(
    snapshots: List, split_range: range, device: torch.device
) -> torch.Tensor:
    y       = torch.cat([snapshots[i].y for i in split_range], dim=0)
    licit   = (y == 0).sum().item()
    illicit = (y == 1).sum().item()
    total   = licit + illicit
    w0      = total / (2.0 * max(licit,   1))
    w1      = total / (2.0 * max(illicit, 1))
    return torch.tensor([w0, w1], dtype=torch.float32, device=device)


def detach_states(states: Optional[Tuple[torch.Tensor, torch.Tensor]]):
    if states is None:
        return None
    return tuple(s.detach() for s in states)


print('Utility functions defined.')

---
## Cell 3 — GCN Norm + WeightEvolver + EvolveGCNHLayer

In [ ]:
def precompute_gcn_norm(
    edge_index:     torch.Tensor,
    n:              int,
    device:         torch.device,
    dtype:          torch.dtype,
    add_self_loops: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if edge_index.numel() == 0:
        if add_self_loops:
            loop_idx = torch.arange(n, device=device)
            row = loop_idx
            col = loop_idx
        else:
            row = torch.empty(0, dtype=torch.long, device=device)
            col = torch.empty(0, dtype=torch.long, device=device)
    else:
        row = edge_index[0]
        col = edge_index[1]
        if add_self_loops:
            loop_idx = torch.arange(n, device=device)
            row = torch.cat([row, loop_idx], dim=0)
            col = torch.cat([col, loop_idx], dim=0)

    deg = torch.zeros(n, device=device, dtype=dtype)
    deg.index_add_(0, row, torch.ones(row.size(0), device=device, dtype=dtype))
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return row, col, norm


def cache_snapshot_graph_norm(snapshots: List) -> None:
    for data in snapshots:
        data._gcn_norm = precompute_gcn_norm(
            edge_index=data.edge_index,
            n=data.x.size(0),
            device=data.x.device,
            dtype=data.x.dtype,
        )


def dense_gcn_propagation(
    x:                torch.Tensor,
    edge_index:       torch.Tensor,
    weight:           torch.Tensor,
    precomputed_norm: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,
    add_self_loops:   bool = True,
) -> torch.Tensor:
    support = x @ weight
    if precomputed_norm is None:
        row, col, norm = precompute_gcn_norm(
            edge_index=edge_index, n=x.size(0),
            device=x.device, dtype=support.dtype,
            add_self_loops=add_self_loops,
        )
    else:
        row, col, norm = precomputed_norm
    if row.numel() == 0:
        return support if add_self_loops else torch.zeros_like(support)
    out = torch.zeros_like(support)
    out.index_add_(0, row, support[col] * norm.unsqueeze(1))
    return out


class WeightEvolver(nn.Module):
    """GRU that evolves the GCN weight matrix over time (EvolveGCN-H)."""
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.hidden_size  = in_channels * out_channels
        self.gru = nn.GRUCell(input_size=in_channels, hidden_size=self.hidden_size)
        self.h0  = nn.Parameter(torch.randn(self.hidden_size) * 0.05)

    def forward(
        self,
        node_embeddings: torch.Tensor,
        prev_hidden:     Optional[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        summary = node_embeddings.mean(dim=0)
        if prev_hidden is None:
            prev_hidden = self.h0
        next_hidden = self.gru(summary, prev_hidden)
        weight_t    = next_hidden.view(self.in_channels, self.out_channels)
        return weight_t, next_hidden


class EvolveGCNHLayer(nn.Module):
    """Single EvolveGCN-H layer: evolves weight via GRU then does GCN propagation."""
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.evolver = WeightEvolver(in_channels, out_channels)

    def forward(
        self,
        x:           torch.Tensor,
        edge_index:  torch.Tensor,
        prev_hidden: Optional[torch.Tensor] = None,
        gcn_norm:    Optional[Tuple]        = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        weight_t, next_hidden = self.evolver(x, prev_hidden)
        out = dense_gcn_propagation(x, edge_index, weight_t, precomputed_norm=gcn_norm)
        return out, next_hidden


print('GCN norm, WeightEvolver, EvolveGCNHLayer defined.')

---
## Cell 4 — EvolveGCNH Model + Splits Dataclass

In [ ]:
@dataclass
class Splits:
    train: range
    val:   range
    test:  range


class EvolveGCNH(nn.Module):
    """
    EvolveGCN-H: two-layer temporal GCN whose weight matrices are evolved
    by a GRU at each snapshot.
    Architecture: 165 -> 64 -> 32 -> 2  (matches Static GCN for fair comparison).
    """
    def __init__(
        self,
        in_channels: int   = 165,
        hidden1:     int   = 64,
        hidden2:     int   = 32,
        dropout:     float = 0.5,
        num_classes: int   = 2,
    ) -> None:
        super().__init__()
        self.layer1     = EvolveGCNHLayer(in_channels, hidden1)
        self.layer2     = EvolveGCNHLayer(hidden1, hidden2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2, num_classes)

    def forward_snapshot(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
        states:     Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        gcn_norm:   Optional[Tuple] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Process a single snapshot.
        states = (h1, h2) -- GRU hidden states for layer1 and layer2.
        Returns logits and updated states to thread to the next snapshot.
        """
        h1_prev = states[0] if states is not None else None
        h2_prev = states[1] if states is not None else None

        # Layer 1
        out1, h1 = self.layer1(x,    edge_index, prev_hidden=h1_prev, gcn_norm=gcn_norm)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)

        # Layer 2 -- no precomputed norm (different embedding dim)
        out2, h2 = self.layer2(out1, edge_index, prev_hidden=h2_prev, gcn_norm=None)
        out2 = F.relu(out2)

        logits = self.classifier(out2)
        return logits, (h1, h2)


print('EvolveGCNH and Splits defined.')
print('Architecture: 165 -> 64 -> 32 -> 2')

---
## Cell 5 — run_sequence, evaluate_auroc, evaluate_f1, train_epoch

In [ ]:
def run_sequence(
    model:     EvolveGCNH,
    snapshots: List,
    seq_range: range,
    device:    torch.device,
    states:    Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
):
    logits_all = []
    labels_all = []
    for i in seq_range:
        data     = snapshots[i]
        gcn_norm = getattr(data, '_gcn_norm', None)
        logits, states = model.forward_snapshot(
            data.x, data.edge_index, states, gcn_norm=gcn_norm
        )
        logits_all.append(logits)
        labels_all.append(data.y)
    return logits_all, labels_all, states


def evaluate_auroc(
    model:        EvolveGCNH,
    snapshots:    List,
    eval_range:   range,
    device:       torch.device,
    warmup_range: Optional[range] = None,
) -> Tuple[float, np.ndarray, np.ndarray]:
    """Evaluate using AUROC -- primary metric for early stopping, matches Static GCN."""
    model.eval()
    states = None
    with torch.no_grad():
        if warmup_range is not None:
            _, _, states = run_sequence(model, snapshots, warmup_range, device, states)
        logits_all, labels_all, _ = run_sequence(model, snapshots, eval_range, device, states)
        y_true = torch.cat(labels_all, dim=0).cpu().numpy()
        probs  = torch.cat(
            [torch.softmax(x, dim=1)[:, 1] for x in logits_all], dim=0
        ).cpu().numpy()
        y_pred = (probs >= 0.5).astype(int)
    auroc = 0.5 if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, probs)
    return auroc, y_true, y_pred


def evaluate_f1(
    model:        EvolveGCNH,
    snapshots:    List,
    eval_range:   range,
    device:       torch.device,
    warmup_range: Optional[range] = None,
) -> Tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    states = None
    with torch.no_grad():
        if warmup_range is not None:
            _, _, states = run_sequence(model, snapshots, warmup_range, device, states)
        logits_all, labels_all, _ = run_sequence(model, snapshots, eval_range, device, states)
        y_true = torch.cat(labels_all, dim=0).cpu().numpy()
        y_pred = torch.cat([x.argmax(dim=1) for x in logits_all], dim=0).cpu().numpy()
    f1 = f1_score(y_true, y_pred, pos_label=1)
    return f1, y_true, y_pred


def train_epoch(
    model:       EvolveGCNH,
    snapshots:   List,
    split_range: range,
    criterion:   nn.Module,
    optimizer:   torch.optim.Optimizer,
    device:      torch.device,
    grad_clip:   float,
    tbptt_steps: int,
) -> float:
    model.train()
    states     = None
    total_loss = 0.0
    for step, i in enumerate(split_range):
        data     = snapshots[i]
        gcn_norm = getattr(data, '_gcn_norm', None)
        optimizer.zero_grad(set_to_none=True)
        logits, states = model.forward_snapshot(
            data.x, data.edge_index, states, gcn_norm=gcn_norm
        )
        loss = criterion(logits, data.y)
        loss.backward()
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
        if tbptt_steps <= 1:
            states = detach_states(states)
        elif (step + 1) % tbptt_steps == 0:
            states = detach_states(states)
    return total_loss / len(split_range)


print('run_sequence, evaluate_auroc, evaluate_f1, train_epoch defined.')

---
## Cell 6 — Config, Data Loading, Model Init

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR      = '/home/blockchain/Desktop/aml_cap/Capstone-AML'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
FIGURES_DIR   = os.path.join(BASE_DIR, 'figures')

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

DATA_PATH  = Path(os.path.join(PROCESSED_DIR, 'snapshots.pt'))
META_PATH  = Path(os.path.join(PROCESSED_DIR, 'meta.json'))
MODEL_PATH = Path(os.path.join(MODELS_DIR,    'evolvegcn_h_best.pt'))

# ── Hyperparameters — matched to Static GCN for fair comparison ───────────────
SEED             = 42
EPOCHS           = 300
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
HIDDEN1          = 64
HIDDEN2          = 32
DROPOUT          = 0.5
GRAD_CLIP        = 1.0
TBPTT_STEPS      = 1
PATIENCE         = 20
VAL_EVERY        = 1
VAL_WARMUP_STEPS = 8

# ── Setup ──────────────────────────────────────────────────────────────────────
set_seed(SEED)
device = get_device()
print(f'Using device : {device}')

snapshots = load_data(DATA_PATH)
print(f'Loaded {len(snapshots)} snapshots')

snapshots = move_snapshots_to_device(snapshots, device)
cache_snapshot_graph_norm(snapshots)
print('Graph normalisation cached.')

with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

num_features = int(meta['num_features'])
print(f'Feature dim  : {num_features}')
assert num_features == 165, f'Expected 165 features, got {num_features}'

splits = Splits(train=range(0, 34), val=range(34, 36), test=range(36, 49))
print('Splits       : train=T1-T34 | val=T35-T36 | test=T37-T49')

class_weights = compute_class_weights(snapshots, splits.train, device)
print(f'Class weights: {class_weights.tolist()}  (licit, illicit)')

model = EvolveGCNH(
    in_channels=num_features,
    hidden1=HIDDEN1,
    hidden2=HIDDEN2,
    dropout=DROPOUT,
    num_classes=2,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters   : {total_params:,}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print('\nModel, criterion, optimizer ready.')

---
## Cell 7 — Training Loop + Test Evaluation

In [ ]:
best_val_auroc = -1.0
best_epoch     = -1
patience_ctr   = 0

warmup_start     = max(splits.train.start, splits.train.stop - VAL_WARMUP_STEPS)
val_warmup_range = range(warmup_start, splits.train.stop)

print(f'Training EvolveGCN-H for up to {EPOCHS} epochs (early stop patience={PATIENCE})')
print(f'Early stopping on: Val AUROC  |  Architecture: 165->{HIDDEN1}->{HIDDEN2}->2')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(
        model=model, snapshots=snapshots, split_range=splits.train,
        criterion=criterion, optimizer=optimizer, device=device,
        grad_clip=GRAD_CLIP, tbptt_steps=TBPTT_STEPS,
    )

    if epoch % VAL_EVERY == 0 or epoch == 1:
        val_auroc, _, _ = evaluate_auroc(
            model=model, snapshots=snapshots, eval_range=splits.val,
            device=device, warmup_range=val_warmup_range,
        )
        marker = ''
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_epoch     = epoch
            patience_ctr   = 0
            torch.save(model.state_dict(), MODEL_PATH)
            marker = '  <- best'
        else:
            patience_ctr += 1
        print(f'Epoch {epoch:03d} | Loss {train_loss:.4f} | Val AUROC {val_auroc:.4f}{marker}')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
            break
    else:
        print(f'Epoch {epoch:03d} | Loss {train_loss:.4f}')

print()
print(f'Best Val AUROC : {best_val_auroc:.4f} at epoch {best_epoch}')
print(f'Model saved to : {MODEL_PATH}')

# ── Test evaluation ────────────────────────────────────────────────────────────
print()
print('Loading best model for test evaluation...')
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))

test_auroc, y_true, y_pred = evaluate_auroc(
    model=model, snapshots=snapshots, eval_range=splits.test,
    device=device, warmup_range=range(0, splits.test.start),
)
test_f1, _, _ = evaluate_f1(
    model=model, snapshots=snapshots, eval_range=splits.test,
    device=device, warmup_range=range(0, splits.test.start),
)

print(f'Test AUROC        : {test_auroc:.4f}')
print(f'Test F1 (illicit) : {test_f1:.4f}')
print()
print(classification_report(y_true, y_pred, digits=4, target_names=['licit', 'illicit']))

# ── Per-snapshot F1 (T37-T49) for drift comparison with Static GCN ────────────
print()
print('Per-snapshot F1 (T37-T49) -- for drift comparison with Static GCN:')
print(f"  {'Snapshot':<10} {'F1':>8}")
print(f"  {'-'*22}")

model.eval()
with torch.no_grad():
    _, _, warmup_states = run_sequence(
        model, snapshots, range(0, splits.test.start), device, None
    )
    states   = warmup_states
    pre_f1s  = []
    post_f1s = []

    for i in splits.test:
        data     = snapshots[i]
        gcn_norm = getattr(data, '_gcn_norm', None)
        logits, states = model.forward_snapshot(
            data.x, data.edge_index, states, gcn_norm=gcn_norm
        )
        preds   = logits.argmax(dim=1).cpu().numpy()
        labels  = data.y.cpu().numpy()
        snap_f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        t       = i + 1
        tag     = ' <- SHUTDOWN' if t == 43 else ''
        print(f'  T{t:<9} {snap_f1:>8.4f}{tag}')
        if t <= 43:
            pre_f1s.append(snap_f1)
        else:
            post_f1s.append(snap_f1)

print()
if pre_f1s:
    print(f'  Pre-T43  mean F1 (T37-T42) : {np.mean(pre_f1s):.4f}')
if post_f1s:
    print(f'  Post-T43 mean F1 (T44-T49) : {np.mean(post_f1s):.4f}')
if pre_f1s and post_f1s:
    print(f'  F1 drop at T43             : {np.mean(pre_f1s) - np.mean(post_f1s):.4f}')